# Agentische mathematische Entdeckung: ein echter Discovery-Lauf

Der Generator bekommt keine fertige Formel und keinen Koeffizienten. Er bekommt nur Beobachtungsdaten und rekonstruiert daraus eine Kandidatenformel. Danach übernimmt der Orchestrator automatisch den gesamten Lebenszyklus: Critic, Gegenbeispielsuche, Refinement, exakte Zertifizierung, Lean-Check und Human Review.

Die Beobachtungen sind der einzige mathematische Input. Die Suche findet bei n=40 ein Gegenbeispiel; anschließend wird die reparierte endliche Aussage 0 <= n < 40 geprüft. Das Notebook nutzt den SQLite-Knowledge-Graph, das hash-verkettete Evidence-Ledger, das Tool-Gateway und lokale Worker-Prozesse aus end_to_end_discovery.py.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import shutil
import sys
import uuid

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "examples", cwd / "Chapter04" / "examples"]
EXAMPLES_DIR = next((p for p in candidates if (p / "end_to_end_discovery.py").is_file()), None)
if EXAMPLES_DIR is None:
    raise FileNotFoundError("Konnte Chapter04/examples/end_to_end_discovery.py nicht finden.")
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

import end_to_end_discovery as discovery
discovery = importlib.reload(discovery)
toolchains = sorted((Path.home() / ".elan" / "toolchains").glob("leanprover--lean4---*/bin/lean"))
LEAN_COMMAND = (
    os.environ.get("LEAN_COMMAND")
    or (str(toolchains[-1]) if toolchains else None)
    or shutil.which("lean")
)
if LEAN_COMMAND is None:
    raise FileNotFoundError("Lean 4 wurde nicht gefunden. Setze LEAN_COMMAND oder installiere Lean mit elan.")

LIVE_LITERATURE = True
WORKERS = 2
AUTO_APPROVE = False
DATABASE_PATH = EXAMPLES_DIR / "artifacts" / f"observation_discovery_{uuid.uuid4().hex[:8]}.sqlite3"
print(f"Beispielcode: {EXAMPLES_DIR}")
print(f"Lean: {LEAN_COMMAND}")
print(f"Literaturrecherche: {'OpenAlex live' if LIVE_LITERATURE else 'explizit offline'}")


## 1. Beobachtungen statt fertiger Vermutung

Für n=0,...,7 sehen wir nur die Werte einer unbekannten quadratischen Folge. Die Formel und insbesondere die Konstante werden nicht an den Orchestrator übergeben.

In [ ]:
observations = [
    (0, 41), (1, 43), (2, 47), (3, 53),
    (4, 61), (5, 71), (6, 83), (7, 97),
]
print("Beobachtungen:", observations)
print("Input für den Generator: nur Datenpaare; keine explizite Formel und keine Koeffizientenbeschriftung.")

## 2. Generator: Kandidatenformel aus den Daten

Konstante zweite Differenzen zeigen eine quadratische Folge. Aus den Daten werden a, b und c der Form a*n^2 + b*n + c rekonstruiert; alle Beobachtungen dienen als Konsistenzcheck.

In [ ]:
def infer_quadratic(observed):
    if len(observed) < 3:
        raise ValueError("Mindestens drei Beobachtungen sind nötig.")
    ordered = sorted(observed)
    xs = [x for x, _ in ordered]
    ys = [y for _, y in ordered]
    if xs != list(range(len(xs))):
        raise ValueError("Es müssen n=0,1,2,... beobachtet sein.")
    first = [ys[i + 1] - ys[i] for i in range(len(ys) - 1)]
    second = [first[i + 1] - first[i] for i in range(len(first) - 1)]
    if len(set(second)) != 1 or second[0] % 2:
        raise ValueError("Keine ganzzahlige quadratische Folge erkannt.")
    a = second[0] // 2
    b = first[0] - a
    c = ys[0]
    if [a * x * x + b * x + c for x in xs] != ys:
        raise ValueError("Die Rekonstruktion erklärt nicht alle Beobachtungen.")
    return a, b, c, first, second

a, b, discovered_constant, first_differences, second_differences = infer_quadratic(observations)
if (a, b) != (1, 1):
    raise ValueError(f"Die lokale Engine erwartet n^2+n+c; entdeckt wurde {a}n^2+{b}n+{discovered_constant}.")
generator_reason = (
    f"Finite differences reconstructed {a}*n^2 + {b}*n + {discovered_constant}; "
    "all supplied observations agree with the candidate."
)
print("Erste Differenzen:", first_differences)
print("Zweite Differenzen:", second_differences)
print("Generator-Kandidat:", f"n^2 + n + {discovered_constant}")


## 3. Ein einziger echter Orchestrator-Lauf

Ab hier werden die Gates nicht mehr manuell nachgebaut. Der Orchestrator ruft den Generator über das Gateway auf, registriert den Claim, prüft Anfangsbeispiele, verteilt die Suche auf Worker, refinanziert nach dem Gegenbeispiel, zertifiziert den endlichen Bereich, ruft Lean auf und hält an der Human-Review-Schranke an.

In [ ]:
def discovered_proposer(_model, _host, _timeout):
    return discovered_constant, generator_reason

outcome = discovery.run(
    model="observation-generator",
    host="local://observation-generator",
    max_n=100,
    database_path=DATABASE_PATH,
    lean_command=LEAN_COMMAND,
    auto_approve=AUTO_APPROVE,
    workers=WORKERS,
    offline_literature=not LIVE_LITERATURE,
    multiprocessing_context="fork" if os.name == "posix" else "spawn",
    proposer=discovered_proposer,
)
print("\nOrchestrator-Ergebnis:", outcome)

## 4. Audit: Was ist tatsächlich passiert?

Nach dem Run öffnen wir die persistierte Datenbank erneut und lesen Endstatus, Ledger-Ereignisse und Knowledge-Graph-Kanten aus.

In [ ]:
audit_db = discovery.DiscoveryDatabase(DATABASE_PATH)
run_row = audit_db.connection.execute(
    "SELECT run_id, outcome, error FROM run_results ORDER BY completed_at DESC LIMIT 1"
).fetchone()
if run_row is None:
    raise RuntimeError("Kein abgeschlossenes Run-Ergebnis gefunden.")
run_id = run_row["run_id"]
print("Run-ID:", run_id)
print("Outcome:", run_row["outcome"])
print("Ledger valide:", audit_db.verify_ledger())

events = audit_db.connection.execute(
    "SELECT sequence, actor, action FROM evidence_events WHERE run_id=? ORDER BY sequence",
    (run_id,),
).fetchall()
print("\nLedger-Ereignisse dieses Runs:")
for event in events:
    print(f"{event['sequence']:03d} | {event['actor']:12} | {event['action']}")

snapshot = audit_db.graph_snapshot(run_id)
print("\nKnowledge Graph:")
print(json.dumps(snapshot, indent=2, ensure_ascii=False))

## 5. Evidence-Gates als überprüfbare Schlussbedingung

Wir prüfen programmatisch, dass die ursprüngliche Vermutung refutiert ist, die verfeinerte Aussage formal verifiziert ist, die Kanten refutes/refines/proves vorhanden sind und die Ledger-Kette intakt ist.

In [ ]:
claims = [node for node in snapshot["nodes"] if node["kind"] == "claim"]
original = next(node for node in claims if node["evidence"] == "REFUTED")
refined = next(node for node in claims if node["evidence"] == "FORMALLY_VERIFIED")
relations = {(edge["relation"], edge["source"], edge["target"]) for edge in snapshot["edges"]}
assert original["statement"].endswith("is prime for every integer n >= 0")
assert refined["statement"].endswith("is prime for integers 0 <= n < 40")
assert any(relation == "refutes" and target == original["id"] for relation, _, target in relations)
assert any(relation == "refines" and target == original["id"] for relation, _, target in relations)
assert any(relation == "proves" and target == refined["id"] for relation, _, target in relations)
assert audit_db.verify_ledger()
print("Alle Evidence-Gates und Auditbedingungen bestanden.")
print("Original:", original["statement"], "->", original["evidence"])
print("Final:", refined["statement"], "->", refined["evidence"])
audit_db.close()
print("Audit-Datenbank geschlossen.")

## 6. Interpretation und Release Review

Formale Prüfung und mathematische Interpretation sind getrennte Eigenschaften. Die Vermutung entsteht aus Beobachtungen, der Orchestrator entscheidet die Übergänge, und die Belege werden von getrennten Rollen erzeugt. Das System erstellt einen Interpretationsentwurf; die menschliche Bestätigung und der zulässige Geltungsbereich werden anschließend im Ledger protokolliert. Lean prüft die endliche reparierte Aussage und die konkrete Faktorisierung; weder Beobachtungen noch endliche Prüfung beweisen, dass n^2+n+41 für alle n prim ist.

In [ ]:
interpretation_outcome, interpretation_approved = discovery.interpretation_review(
    DATABASE_PATH,
    run_id=run_id,
    auto=False,
)
print("Interpretation Review:", "approved" if interpretation_approved else "not approved")
print("Run-Outcome:", interpretation_outcome)
